# Agentic Report

### No Claude Code Used!

In [1]:
from typing import TypedDict, List, Optional, Annotated
from langgraph.graph.message import add_messages
from langchain_core.tools import tool, InjectedToolArg
from langgraph.graph import StateGraph, END
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from dotenv import load_dotenv
import PyPDF2
import os
import openai
import warnings
import re
from book_agent import book_lookup
from python_tool import execute_python_code
from prompts import *
import pandas as pd
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage

warnings.filterwarnings('ignore')
print("✅ All libraries imported successfully!")

✅ Helper functions with citation support defined!
✅ All libraries imported successfully!


In [2]:
# Option 1: Load from .env file
load_dotenv()

# Verify API key is set
if os.getenv("OPENAI_API_KEY"):
    print("✅ API key loaded successfully!")
else:
    print("❌ API key not found. Please set OPENAI_API_KEY")

✅ API key loaded successfully!


In [3]:
class AgentState(TypedDict):
    task: str
    data: pd.DataFrame
    messages: Annotated[List, add_messages]

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.utils.function_calling import convert_to_openai_function

tools = [book_lookup, execute_python_code]
functions = [convert_to_openai_function(f) for f in tools]

model = ChatOpenAI(
    model="gpt-4o",
    temperature=0.05,
    max_tokens=None,
    timeout=None,
    max_retries=2).bind_tools(functions)

In [12]:
def load_data_node(state: AgentState):
    """
    Agent loads data and gets comfortable with what it is looking at. Creates_graphs as needed
    """
    messages = AgentState.get("messages", [])
    
    if len(messages) > 1:
        last_msg = messages[-1]
        if not(hasattr(last_msg, 'tool_calls') and last_msg.too_calls) and \
            not last_msg.additional_kwargs.get("funciton_call"):
            print(" Tools already executed, ending")
        return AgentState

    print(" First run - loading data and generating code")
    file_path = "aggregated_features_addresses.csv"
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
    else:
        raise FileNotFoundError(f"{file_path} not found")

    response = model.invoke([
        SystemMessage(content=LOAD_DATA_PROMPT),
        HumanMessage(content=f"Here's the data:n{df.head()}\\nColumns: {df.columns.tolist()}\n\nWrite code to generate relevant graphs of this data and save them as PNG files.")
    ])

    print(f"DEBUG: Model response type: {type(response)}")
    print(f"DEBUG: Has tool_calls: {hasattr(response, 'tool_calls') and bool(response.tool_calls)}")
    print(f"DEBUG: Has function_call: {bool(response.additional_kwargs.get('function_call'))}")
    
    new_state =  {"messages":messages + [response],
           "data": df
           }

    print(f"DEBUG: Returning state with {len(new_state['messages'])} messages")

    return new_state

In [13]:
def organize_task_node(state: AgentState):
    """
    Agent takes in task and data and figures out what is needed to write the report.
    Comes up with query to ask the book.
    """
    messages = [

    ]

In [14]:
def write_report(state: AgentState):
    """
    Agent writes the report
    """

In [15]:
def tool_execution_node(state):
    """Custom tool node with debug output"""
    print("DEBUG: Entered tool_execution_node")
    messages = state["messages"]
    last_message = messages[-1]

    print(f"DEBUG: Last message has {len(last_message.tool_calls)} tool calls")

    # Execute using TOolNode
    tool_node = ToolNode([execute_python_code])
    result = tool_node.invoke(state)

    print(f"DEBUG: Tool exeuction completed, result keys: {result.keys()}")
    if "messages" in result:
        print(f"DEUBG: Tool returned {len(result['messages'])} messages")
        for msg in result['messages']:
            print(f"DEBUG: Message type: {type(msg)}, content preview: {str(msg.content)[:200]}")
    return result

In [16]:
# Add nodes
agent_workflow = StateGraph(AgentState)

agent_workflow.add_node("load_data", load_data_node) ####
agent_workflow.add_node("tools", tool_execution_node)
#agent_workflow.add_node("organize_task", organize_task_node) ###
#agent_workflow.add_node("research", research_node) ####
#agent_workflow.add_node("write_report", write_report_node) ####

# graph structure
agent_workflow.set_entry_point("load_data")

def should_continue(state):
    last_message = state["messages"][-1]
    print(f"DEBUG should_continue: message type = {type(last_message)}")
    print(f"DEBUG should_continue: has tool_calls = {hasattr(last_message, 'tool_calls') and bool(last_message.tool_calls)}")
    print(f"DEBUG should_continue: has funciton_call = {bool(last_message.additional_kwargs.get('function_call'))}")
    if last_message.tool_calls or last_message.additional_kwargs.get("function_call"):
        print("DEBUG should_continue: Routing to TOOLS")
        return "tools"
    print("DEBUG should_continue: Routing to END")
    return END
    
agent_workflow.add_conditional_edges(
    "load_data",
    should_continue
)
agent_workflow.add_edge("tools",END) #"organize_task"
#agent_workflow.add_edge("load_data","research")
#agent_workflow.add_edge("research","write_report")
#agent_workflow.add_edge("load_data", END) # write_report
graph = agent_workflow.compile()

In [17]:
result = graph.invoke({
    "messages": []})

 First run - loading data and generating code
DEBUG: Model response type: <class 'langchain_core.messages.ai.AIMessage'>
DEBUG: Has tool_calls: True
DEBUG: Has function_call: False
DEBUG: Returning state with 1 messages
DEBUG should_continue: message type = <class 'langchain_core.messages.ai.AIMessage'>
DEBUG should_continue: has tool_calls = True
DEBUG should_continue: has funciton_call = False
DEBUG should_continue: Routing to TOOLS
DEBUG: Entered tool_execution_node
DEBUG: Last message has 1 tool calls
DEBUG: Tool exeuction completed, result keys: dict_keys(['messages'])
DEUBG: Tool returned 1 messages
DEBUG: Message type: <class 'langchain_core.messages.tool.ToolMessage'>, content preview: Code executed successfully


In [11]:
result

{'data':                                        address  normal_first_txn_dt  \
 0   0xB5359336b305C8db8a89524f115528cF7aD89eA2  2026-04-23 18:07:23   
 1   0x7174d846b27fd468853303895aBb8dbE95E44808  2026-01-31 02:30:35   
 2   0x0A76D0C88683Dc3AcFEb4DEdF47064a9B44e8699  2024-08-26 17:22:23   
 3   0x641C0882b0De34308db18310DC080B736A673bA1  2026-03-07 01:31:47   
 4   0x30a5cbe88A7fE348fc2902e98C38bba36fA614D7  2026-01-08 19:53:11   
 5   0x02f03E4AEfc35dF901AC734DEC9e028A708A4Ad9  2025-01-11 12:26:35   
 6   0xd4AA54A29e359fBe037d8490bF37F2A23256A866  2026-02-11 17:41:23   
 7   0x6f92dE21f6E34b1bcAAD7A33dE5B3a5e370eEf11  2026-03-26 03:00:11   
 8   0xddbd2b932c763ba5b1b7ae3b362eac3e8d40121a  2015-08-07 11:45:53   
 9   0x6a7b8d6032640e37c1db7b44378fa7bc063002f9  2025-03-09 09:21:59   
 10  0xebDd09fb0786249F160d72101b8802A191b9899D  2025-10-02 18:06:35   
 11  0x594F34f1DF5bAEe31b1Cf6B1574C8957891FcF42  2025-04-25 21:28:47   
 12  0x55F9e98e2fc079F8FAbe2109eA539E2B578aD794  2024-09